# SeqTrainer Tutorial 04: End-to-end mini CNN classifier

This notebook compiles the prior tutorials into one compact training workflow:
1) build a dataset from SBOL files, 2) engineer fixed-length one-hot DNA tensors, 3) split into train/val/test, and 4) train a small CNN + classification head for **10 cycles**.

## Goal

This is a short demonstration run, not full model training.

To scale up later, just increase `NUM_CYCLES` (or tune batch size / model width).

In [ ]:
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from seqtrainer.data.sbol import build_dataset_from_files
from seqtrainer.data.materialized import MaterializedDataset
from seqtrainer.transforms.dna import one_hot_encode, pad_or_trim

print("Imports loaded")

## 1) Load SBOL data and create a binary label

For tutorial purposes, we derive a class label by thresholding `target` at the median.

In [ ]:
base = Path("data/sbol_data")
files = sorted(base.glob("sample_design_*.xml"))[:40]
df = build_dataset_from_files(files)

if df.empty:
    raise RuntimeError("No rows were materialized from SBOL inputs.")

threshold = float(df["target"].median())
df["label"] = (df["target"] >= threshold).astype(int)

print(f"Rows: {len(df)} | threshold: {threshold:.4f}")
df[["sequence", "target", "label"]].head()

## 2) DNA preprocessing (fixed length + one-hot)

CNNs need consistent sequence length, so we pad/trim each sequence.

In [ ]:
SEQ_LEN = 120

fixed_sequences = [pad_or_trim(seq, length=SEQ_LEN) for seq in df["sequence"].tolist()]
X = one_hot_encode(fixed_sequences)  # shape: [N, L, C]
y = df["label"].to_numpy(dtype=np.int64)

# PyTorch Conv1d expects [N, C, L]
X_t = torch.tensor(np.transpose(X, (0, 2, 1)), dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)

X_t.shape, y_t.shape

## 3) Train/val/test split using `MaterializedDataset`

In [ ]:
examples = [{"idx": i, "label": int(label)} for i, label in enumerate(y)]
materialized = MaterializedDataset(examples, metadata={"tutorial": "cnn_demo"})
train_ds, val_ds, test_ds = materialized.train_val_test_split(0.7, 0.15, 0.15, seed=42)

def to_index_tensor(split):
    return torch.tensor([row["idx"] for row in split.examples], dtype=torch.long)

train_idx, val_idx, test_idx = map(to_index_tensor, (train_ds, val_ds, test_ds))
len(train_idx), len(val_idx), len(test_idx)

## 4) Build dataloaders

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(TensorDataset(X_t[train_idx], y_t[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_t[val_idx], y_t[val_idx]), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_t[test_idx], y_t[test_idx]), batch_size=BATCH_SIZE)

len(train_loader), len(val_loader), len(test_loader)

## 5) Define a compact CNN backbone + classification head

In [ ]:
class TinyDNACNN(nn.Module):
    def __init__(self, channels: int = 5, n_classes: int = 2):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = TinyDNACNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

## 6) Train for 10 cycles (easy to increase)

Change `NUM_CYCLES` to run longer training.

In [ ]:
NUM_CYCLES = 10

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, total_correct / total

for cycle in range(1, NUM_CYCLES + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(
        f"cycle={cycle:02d} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
    )

## 7) Quick test-set check

In [ ]:
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"test_loss={test_loss:.4f} test_acc={test_acc:.3f}")